# Denoise Emotion Vectors


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"  # must be before import torch

import json
import torch
import numpy as np
import torch.nn.functional as F
from pathlib import Path
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import sys
sys.path.append("..")
from config import DEFAULT_MODEL, TARGET_LAYER, TOKEN_START, BATCH_SIZE, VECTORS_OUT

NEUTRAL_JSONL    = "data/neutral_depositions.jsonl"
DENOISED_OUT     = "data/emotion_vectors_denoised.pt"
VARIANCE_TARGET  = 0.50

print(f"Layer        : {TARGET_LAYER}")
print(f"Token start  : {TOKEN_START}")
print(f"Variance target: {VARIANCE_TARGET:.0%}")

## Load Model


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

## Extract Neutral Activations


In [ ]:
# Load neutral deposition excerpts
neutral_texts = []
with open(NEUTRAL_JSONL) as f:
    for line in f:
        line = line.strip()
        if line:
            neutral_texts.append(json.loads(line)["text"])

print(f"Loaded {len(neutral_texts)} neutral deposition excerpts")

# Helper functions (same as extraction notebook)
def mean_pool_from(hidden, start):
    sliced = hidden[start:]
    if sliced.shape[0] == 0:
        sliced = hidden
    return sliced.mean(dim=0)

@torch.inference_mode()
def encode_texts(texts, layer, token_start, batch_size):
    all_vecs = []
    input_device = model.model.embed_tokens.weight.device
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=512).to(input_device)
        out = model(**enc, output_hidden_states=True, use_cache=False)
        layer_h = out.hidden_states[layer].float().cpu()
        mask    = enc["attention_mask"].cpu()
        for b in range(layer_h.shape[0]):
            seq_len = mask[b].sum().item()
            h   = layer_h[b, :seq_len, :]
            vec = mean_pool_from(h, token_start)
            all_vecs.append(vec)
    return torch.stack(all_vecs, dim=0)

print("Encoding neutral deposition excerpts...")
neutral_acts = encode_texts(
    neutral_texts,
    layer=TARGET_LAYER,
    token_start=TOKEN_START,
    batch_size=BATCH_SIZE,
)
print(f"Neutral activations shape: {neutral_acts.shape}")

## PCA on Neutral Activations


In [ ]:
# Center neutral activations before PCA
neutral_np   = neutral_acts.numpy()
neutral_mean = neutral_np.mean(axis=0, keepdims=True)
neutral_centered = neutral_np - neutral_mean

# Fit PCA — keep enough components for the full explained variance curve
n_components = min(len(neutral_texts), neutral_acts.shape[1])
pca = PCA(n_components=min(n_components, 200))  # cap at 200 for speed
pca.fit(neutral_centered)

# Find K: smallest number of components explaining >= VARIANCE_TARGET
cumvar = np.cumsum(pca.explained_variance_ratio_)
K = int(np.searchsorted(cumvar, VARIANCE_TARGET)) + 1

print(f"Components to explain {VARIANCE_TARGET:.0%} variance: K = {K}")
print(f"Cumulative variance at K={K}: {cumvar[K-1]:.1%}")
print()
print("Top 10 component explained variance:")
for i, v in enumerate(pca.explained_variance_ratio_[:10]):
    print(f"  PC{i+1:>2}: {v:.2%}  (cumulative: {cumvar[i]:.2%})")


## Project Out Neutral Components


In [ ]:
# Top K principal components — shape (K, 4096)
neutral_pcs = torch.tensor(pca.components_[:K], dtype=torch.float32)  # (K, D)

# Load raw emotion vectors
vecs_dict = torch.load(VECTORS_OUT, map_location="cpu", weights_only=False)
emotions  = sorted(vecs_dict.keys())

def project_out(v, pcs):
    """Remove components of v along each direction in pcs. v: (D,), pcs: (K, D)"""
    for pc in pcs:
        v = v - (v @ pc) * pc
    return v

emotion_vectors_denoised = {}
print(f"Projecting out {K} neutral PCs from {len(emotions)} emotion vectors...
")
print(f"  {'Emotion':<20} {'Norm before':>12}  {'Norm after':>10}  {'Change':>8}")
print("  " + "-" * 55)

for emotion in emotions:
    v_raw      = vecs_dict[emotion].float()
    v_denoised = project_out(v_raw.clone(), neutral_pcs)
    emotion_vectors_denoised[emotion] = v_denoised

    norm_before = v_raw.norm().item()
    norm_after  = v_denoised.norm().item()
    change      = (norm_after - norm_before) / norm_before * 100
    print(f"  {emotion:<20} {norm_before:>12.4f}  {norm_after:>10.4f}  {change:>+7.1f}%")


## Save Denoised Vectors


In [ ]:
Path(DENOISED_OUT).parent.mkdir(parents=True, exist_ok=True)
torch.save(emotion_vectors_denoised, DENOISED_OUT)
torch.save(neutral_pcs, "data/neutral_pcs.pt")

print(f"Saved denoised vectors -> {DENOISED_OUT}")
print(f"Saved neutral PCs      -> data/neutral_pcs.pt")
print(f"Vector shape: {next(iter(emotion_vectors_denoised.values())).shape}")


## Validation on Held-Out Stories


In [ ]:
from collections import defaultdict

N_HOLDOUT    = 40
STORIES_JSONL = "data/stories.jsonl"

# Load held-out stories
all_stories = defaultdict(list)
with open(STORIES_JSONL) as f:
    for line in f:
        line = line.strip()
        if line:
            rec = json.loads(line)
            all_stories[rec["emotion"]].append(rec["text"])

eval_pairs = [(t, e) for e, texts in all_stories.items() for t in texts[-N_HOLDOUT:]]
texts  = [p[0] for p in eval_pairs]
labels = [p[1] for p in eval_pairs]

# Build normalised denoised matrix
mat_d = torch.stack([emotion_vectors_denoised[e] for e in emotions], dim=0).float()
mat_d = F.normalize(mat_d, dim=1)
emotion_to_idx = {e: i for i, e in enumerate(emotions)}

print(f"Encoding {len(texts)} held-out stories...")
activations = encode_texts(texts, layer=TARGET_LAYER, token_start=TOKEN_START, batch_size=BATCH_SIZE)
activations = F.normalize(activations.float(), dim=1)
scores = activations @ mat_d.T

# Compute results
def rank_of_target(score_row, target_idx):
    return score_row.argsort(descending=True).tolist().index(target_idx) + 1

all_ranks = []
results   = {}
for emotion in emotions:
    target_idx = emotion_to_idx[emotion]
    idxs  = [i for i, l in enumerate(labels) if l == emotion]
    ranks = [rank_of_target(scores[i], target_idx) for i in idxs]
    all_ranks.extend(ranks)
    results[emotion] = {
        "top1_acc":  sum(r == 1 for r in ranks) / len(ranks),
        "top3_acc":  sum(r <= 3 for r in ranks) / len(ranks),
        "mean_rank": float(np.mean(ranks)),
    }

random_baseline = 1 / len(emotions)
overall_top1 = sum(r == 1 for r in all_ranks) / len(all_ranks)
overall_top3 = sum(r <= 3 for r in all_ranks) / len(all_ranks)

print(f"\nDenoised vectors — held-out story validation")
print(f"Overall top-1 : {overall_top1:.1%}  (random baseline: {random_baseline:.1%})")
print(f"Overall top-3 : {overall_top3:.1%}")
print(f"Mean rank     : {np.mean(all_ranks):.1f} / {len(emotions)}")
print()
print(f"  {'Emotion':<20} {'Top1':>6}  {'Top3':>6}  {'MeanRank':>9}")
print("  " + "-" * 45)
for emotion, r in sorted(results.items(), key=lambda x: -x[1]['top1_acc']):
    print(f"  {emotion:<20} {r['top1_acc']:>6.1%}  {r['top3_acc']:>6.1%}  {r['mean_rank']:>9.1f}")
